In [ ]:
!pip install jiwer
!pip install evaluate
!pip install pytorch_lightning

In [ ]:
!pip install gdown

In [ ]:
!gdown 1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ

Downloading...
From (original): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ
From (redirected): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ&confirm=t&uuid=793ebe6e-92c7-4d83-bdc3-4385640c2990
To: /content/dataset.zip
100% 9.12G/9.12G [01:52<00:00, 81.4MB/s]


In [8]:
!unzip dataset.zip -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/toronto_6/toronto_6_38.wav  
  inflating: dataset/toronto_6/toronto_6_39.wav  
  inflating: dataset/toronto_6/toronto_6_40.wav  
  inflating: dataset/toronto_6/toronto_6_41.wav  
  inflating: dataset/toronto_6/toronto_6_42.wav  
  inflating: dataset/toronto_6/toronto_6_43.wav  
  inflating: dataset/toronto_6/toronto_6_44.wav  
  inflating: dataset/toronto_6/toronto_6_45.wav  
  inflating: dataset/toronto_6/toronto_6_46.wav  
  inflating: dataset/toronto_6/toronto_6_47.wav  
  inflating: dataset/toronto_6/toronto_6_48.wav  
  inflating: dataset/toronto_6/toronto_6_49.wav  
  inflating: dataset/toronto_6/toronto_6_50.wav  
  inflating: dataset/toronto_6/toronto_6_51.wav  
  inflating: dataset/toronto_6/toronto_6_52.wav  
  inflating: dataset/toronto_6/toronto_6_53.wav  
  inflating: dataset/toronto_6/toronto_6_54.wav  
  inflating: dataset/toronto_6/toronto_6_55.wav  
  inflating: dataset/toronto_6/toronto_6_56.wav  

In [10]:
!ls /content/dataset

labels.jsonl  toronto_139  toronto_166	toronto_35  toronto_55	toronto_81
toronto_0     toronto_14   toronto_17	toronto_36  toronto_58	toronto_83
toronto_100   toronto_144  toronto_170	toronto_37  toronto_59	toronto_84
toronto_101   toronto_145  toronto_172	toronto_38  toronto_6	toronto_85
toronto_11    toronto_148  toronto_18	toronto_4   toronto_60	toronto_86
toronto_12    toronto_15   toronto_187	toronto_42  toronto_62	toronto_87
toronto_123   toronto_150  toronto_188	toronto_43  toronto_66	toronto_89
toronto_127   toronto_153  toronto_2	toronto_44  toronto_67	toronto_9
toronto_128   toronto_155  toronto_21	toronto_45  toronto_68	toronto_92
toronto_130   toronto_156  toronto_23	toronto_46  toronto_7	toronto_93
toronto_133   toronto_157  toronto_25	toronto_49  toronto_72	toronto_94
toronto_134   toronto_159  toronto_26	toronto_5   toronto_74	toronto_95
toronto_135   toronto_16   toronto_27	toronto_50  toronto_75	toronto_97
toronto_136   toronto_161  toronto_3	toronto_53  toronto_77
tor

In [11]:
import os
import re
import json
import random
import logging
import evaluate
import torch
import torchaudio
import pytorch_lightning as pl

from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass
from pytorch_lightning import Trainer
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [12]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [13]:
AUDIO_DIR = Path("/content")

TEST_IDS = [
    "toronto_27", "toronto_46", "toronto_42", "toronto_37",
    "toronto_43", "toronto_157", "toronto_9", "toronto_156",
    "toronto_7", "toronto_123", "toronto_54", "toronto_67",
    "toronto_62", "toronto_81", "toronto_134", "toronto_148",
    "toronto_21", "toronto_135", "toronto_166", "toronto_58",
]

In [14]:
def split_data(items):
    train, val, test = [], [], []

    for item in items:
        file_id = item["path"].split("/")[-2]  # toronto_157

        if file_id in TEST_IDS:
            test.append(item)
        else:
            train.append(item)

    random.shuffle(train)
    val = train[:int(0.05 * len(train))]
    train = train[int(0.05 * len(train)):]

    return train, val, test

In [15]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\sа-яіїєґ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [16]:
def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = []
    missing = 0

    for audio_path, text in data.items():
        if os.path.exists(AUDIO_DIR / audio_path):
            items.append({
                "path": audio_path,
                "text": text
            })
        else:
            missing += 1

    print(f"Loaded: {len(items)}")
    print(f"Missing files skipped: {missing}")

    return items

In [17]:
class TorontoDataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        audio, sr = torchaudio.load(AUDIO_DIR / item["path"])
        audio = audio.mean(dim=0)  # mono

        if sr != 16000:
            audio = torchaudio.functional.resample(audio, sr, 16000)

        text = normalize_text(item["text"])

        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            # padding="longest",
        )

        # labels = self.processor.tokenizer(
        #     text,
        #     return_tensors="pt"
        # ).input_ids
        labels = self.processor(
            text=text,
            return_tensors="pt"
        ).input_ids

        return {
            "input_features": inputs.input_features[0],
            "labels": labels[0]
        }

In [18]:
@dataclass
class DataCollator:
    processor: any

    def __call__(self, batch):
        input_features = [b["input_features"] for b in batch]
        labels = [b["labels"] for b in batch]

        input_features = torch.nn.utils.rnn.pad_sequence(
            input_features, batch_first=True
        )

        attention_mask = torch.ones_like(input_features[:, :, 0])

        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [19]:
class WhisperLightning(pl.LightningModule):
    def __init__(self, model_name, processor, lr=1e-5):
        super().__init__()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

        self.model.generation_config.suppress_tokens = []
        self.model.generation_config.forced_decoder_ids = None

        self.processor = processor
        self.lr = lr

    def training_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("train_loss", out.loss, prog_bar=True, on_step=True, on_epoch=True)
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("val_loss", out.loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

In [20]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="uk",
    task="transcribe",
)

items = load_labels(AUDIO_DIR / "dataset/labels.jsonl")
train_items, val_items, test_items = split_data(items)

train_ds = TorontoDataset(train_items, processor)
val_ds = TorontoDataset(val_items, processor)

collator = DataCollator(processor)

train_loader = DataLoader(
    train_ds, batch_size=8, num_workers=4, shuffle=True, collate_fn=collator,
)
val_loader = DataLoader(
    val_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

model = WhisperLightning("openai/whisper-tiny", processor)
model.train()

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="gpu",
    # precision="bf16-mixed",
    enable_progress_bar=True,
    log_every_n_steps=1,
)

trainer.fit(model, train_loader, val_loader)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Trainable params: 37.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 37.8 M                                                                                               
Total estimated model params size (MB): 151                                                                        
Modules in train mode: 126                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will 
create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [21]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def evaluate_model(model, dataloader, processor):
    preds, refs = [], []

    model.eval()
    for batch in tqdm(dataloader):
        with torch.no_grad():
            pred_ids = model.model.generate(
                batch["input_features"].to(model.device),
                language="uk",
                task="transcribe",
            )

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_str = processor.batch_decode(labels, skip_special_tokens=True)

        preds.extend(pred_str)
        refs.extend(ref_str)

    return preds, refs

In [23]:
# 1. для інференсу
model.model.save_pretrained("whisper_full")
model.processor.save_pretrained("whisper_full")

# 2. для resume training
trainer.save_checkpoint("whisper_lightning.ckpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:pytorch_lightning.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


In [26]:
next(model.model.parameters()).device

device(type='cuda', index=0)

In [25]:
model.to("cuda")

WhisperLightning(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 384)
        (layers): ModuleList(
          (0-3): 4 x WhisperEncoderLayer(
            (self_attn): WhisperAttention(
              (k_proj): Linear(in_features=384, out_features=384, bias=False)
              (v_proj): Linear(in_features=384, out_features=384, bias=True)
              (q_proj): Linear(in_features=384, out_features=384, bias=True)
              (out_proj): Linear(in_features=384, out_features=384, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (fc2): Linea

In [27]:
test_ds = TorontoDataset(test_items, processor)
test_loader = DataLoader(
    test_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

preds, refs = evaluate_model(model, test_loader, processor)

indices = random.sample(range(len(preds)), 10)
print("Sample predictions:")
for idx in indices:
    print(f"Ref: {refs[idx]}")
    print(f"Pred: {preds[idx]}")
    print("-" * 20)

wer = wer_metric.compute(predictions=preds, references=refs)
cer = cer_metric.compute(predictions=preds, references=refs)
print(f"WER: {wer}, CER: {cer}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|          | 0/667 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100

Sample predictions:
Ref: директор назвав самогоноваріння виробничою потребою все правильно самогон село ж називається ли пянка
Pred: правляється у пояснювальні посадовцям директор назвав самого новарійня виробний чої потребуювсе правильно самогон це
--------------------
Ref: класно1 класно2 класно3 класно4 класно 5 класно 6 класно7 перемотка перелічення
Pred: клашно дивім класно 23 класно 4 класно 5 класно 6 класно 7 класно
--------------------
Ref: з оголошення імпічменту ми починаємо процедуру імпічмента президенту а ви звернули увагу як
Pred: заголошення інпічменту ми починаємо процедуру інпічмента президенту а ви звернули увагу як
--------------------
Ref: іде дощ і це так мало схоже на різдво
Pred: і до тож і цві так мало схож і нарістло
--------------------
Ref: слава ісу ви дивитесь програму 0 я є її ведучий майкл щур але я зараз у армії
Pred: слава ісу ви дивитесь програму 0 я її ведучий майкл щур але зараз у армі
--------------------
Ref: навіть бога come on локі ж вбив танос 